In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd


PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
# Configuration flags
USE_FRACTION: bool = False
SELECTED_SURFACE: str = "all"  # Options: "hard", "clay", or "all"

# Derived configuration
USE_FRACTION_STR: str = "ratio" if USE_FRACTION else "num"

In [ ]:
# Base raw data directory
DATA_FOLDER = Path("../../../data")

# Subdirectories processed data
DATA_FOLDER_PROCESSED = DATA_FOLDER / "02_processed"

# Processed file (depends on selected surface)
FILE_CLEANING = DATA_FOLDER_PROCESSED / f"cleaning__processed_value__{SELECTED_SURFACE}.csv"
FILE_FEATURE  = DATA_FOLDER_PROCESSED / f"/feature__{USE_FRACTION_STR}_value__{SELECTED_SURFACE}.csv"

# tmp file
FILE_TMP= DATA_FOLDER_PROCESSED / "tmp.csv"

## I. Read cleaning file

In [ ]:
# FEATURE Preprocessed
df = pd.read_csv(FILE_CLEANING)

In [ ]:
print(df.shape)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

## II. RANTING FEATURE

In [ ]:
from prediction_tennis.src.preprocessing.features.simple_ranking_features import compute_simple_ranking, compute_transformed_winloss_rankings  
from prediction_tennis.src.preprocessing.features.elo_ranking_features import compute_momentum_elo_rankings, compute_elo_rankings
from prediction_tennis.src.preprocessing.features.glicko_ranking_features import compute_glicko_ratings      
from prediction_tennis.src.preprocessing.features.trueskill_ranking_features import compute_trueskill_ratings

# from prediction_tennis.src.preprocessing.features.time_decay_ratings import compute_time_decay
from prediction_tennis.src.preprocessing.features.compute_rating_movement import compute_rating_movement

In [ ]:
# Get the unique surface groups.
surface_groups = df['surface_group'].dropna().unique()

###  A. SIMPLE RANKING

#### 1. SIMPLE - Surface

In [ ]:
type_feature = "simple_surface"
feature_p1 = f"{type_feature}_p1"
feature_p2 = f"{type_feature}_p2"
feature_diff = f"{type_feature}_diff"

# Initialize new columns in the DataFrame
df[[feature_p1, feature_p2, feature_diff]] = None

for surface in surface_groups:
    # Filter DataFrame for the current surface group
    df_surface = df[df['surface_group'] == surface].copy().reset_index(drop=True)
    
    # Compute the rankings for the current surface group
    rankings = compute_simple_ranking(matches_df=df_surface, surface=surface)
    
    # Assign rankings back to the original DataFrame
    df.loc[df['surface_group'] == surface, [feature_p1, feature_p2]] = rankings
    
    # Compute and assign the difference between player1 and player2 rankings
    df.loc[df['surface_group'] == surface, feature_diff] = rankings[:, 0] - rankings[:, 1]


#### 2. SIMPLE - All

In [ ]:
# Define the ranking methods and corresponding step values
ranking_configs = [
    ("power", 7.1, 4.1),
    ("log"  , 4.6, 2.1),
    ("exp"  , 3.1, 2.6)
]

for method, win_step, lose_step in ranking_configs:
    # Compute simple rankings for the current method
    match_rankings = compute_simple_ranking(
        matches_df=df.copy(),
        win_step=win_step,
        lose_step=lose_step,
        transformation_method=method
    )

    # Define feature names
    feature_p1 = f"simple_{method}_p1"
    feature_p2 = f"simple_{method}_p2"
    feature_diff = f"simple_{method}_diff"

    # Assign rankings and difference to the DataFrame
    df[[feature_p1, feature_p2]] = match_rankings
    df[feature_diff] = df[feature_p1] - df[feature_p2]

In [ ]:
# Define the ranking methods and corresponding step values
ranking_configs = [
    ("power", 7.1, 4.1),
    ("log", 1.0, 1.0),
    ("exp", 1.0, 1.0)
]

for method, win_step, lose_step in ranking_configs:
    # Compute transformed win-loss rankings for the current method
    match_rankings = compute_transformed_winloss_rankings(
        matches_df=df.copy(),
        win_step=win_step,
        lose_step=lose_step,
        transformation_method=method
    )

    # Define feature names
    feature_p1 = f"simple_all_{method}_p1"
    feature_p2 = f"simple_all_{method}_p2"
    feature_diff = f"simple_all_{method}_diff"

    # Assign rankings and differences to the DataFrame
    df[[feature_p1, feature_p2]] = match_rankings
    df[feature_diff] = df[feature_p1] - df[feature_p2]


#### 3 SIMPLE - Time Decay 

In [ ]:
# for method in ["log", "exp", "power"]:
#     type_feature = f"simple_{method}"
#     match_elo_rankings = compute_time_decay(df_matches=df.copy(), lambda_=0.05, col_prefix=f"{type_feature}")

#     type_feature = f"simple_{method}_time"
#     feature1   = f"{type_feature}_p1"
#     feature2   = f"{type_feature}_p2"
#     minus_name = f"{type_feature}_diff"

#     df[[feature1, feature2]] = match_elo_rankings
#     df[minus_name] = df[feature1] - df[feature2]

# # print(f"{match_elo_rankings.min()} - {match_elo_rankings.max()}")

#### 4 SIMPLE - Movement

In [ ]:
# Define methods and last_match intervals
methods = ["log", "exp", "power"]
last_matches = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30]


In [ ]:
new_features = []

for method in methods:
    rating_prefix = f"simple_{method}"

    for last_match in last_matches:
        # Compute rating movements based on the last N matches
        match_movements = compute_rating_movement(
            matches_df=df.copy(),
            lookback_matches=last_match,
            rating_prefix=rating_prefix
        )

        # Define feature names
        feature_p1   = f"{rating_prefix}_mov_{last_match}_p1"
        feature_p2   = f"{rating_prefix}_mov_{last_match}_p2"
        feature_diff = f"{rating_prefix}_mov_{last_match}_diff"

        # Convert NumPy array to DataFrame with proper column names
        match_df = pd.DataFrame(match_movements, columns=[feature_p1, feature_p2])

        # compute difference
        match_df[feature_diff] = match_movements[:, 0] - match_movements[:, 1]

        new_features.append(match_df)

# Concatenate all new columns to the original DataFrame at once
df = pd.concat([df] + new_features, axis=1)

In [ ]:
# Container for all new features
all_new_features = []

for method in methods:
    rating_prefix_all = f"simple_all_{method}"

    for last_match in last_matches:
        # Compute rating movements based on the last N matches
        match_movements = compute_rating_movement(
            matches_df=df.copy(),
            lookback_matches=last_match,
            rating_prefix=rating_prefix_all
        )
        
        # Define feature names
        feature_p1   = f"{rating_prefix}_mov_{last_match}_p1"
        feature_p2   = f"{rating_prefix}_mov_{last_match}_p2"
        feature_diff = f"{rating_prefix}_mov_{last_match}_diff"

        # Convert NumPy array to DataFrame with proper column names
        match_df = pd.DataFrame(match_movements, columns=[feature_p1, feature_p2])

        # compute difference
        match_df[feature_diff] = match_movements[:, 0] - match_movements[:, 1]

        all_new_features.append(match_df)

# Concatenate all new columns to the original DataFrame at once
df = pd.concat([df] + all_new_features, axis=1)

#### 5 SIMPLE - Save df

In [ ]:
# Save
df.to_csv(FILE_TMP)

### B. ELO RANKING

In [ ]:
df = pd.read_csv(FILE_TMP)

#### 1. ELO - Surface

In [ ]:

type_feature = "elo_surface"
feature_p1 = f"{type_feature}_p1"
feature_p2 = f"{type_feature}_p2"
feature_diff = f"{type_feature}_diff"

# Initialize new columns in the DataFrame
df[[feature_p1, feature_p2, feature_diff]] = None

for surface in surface_groups:
    # Filter the DataFrame for the current surface group.
    df_surface = df[df['surface_group'] == surface].copy().reset_index(drop=True)

    # Compute the ELO rankings for the current surface group.
    rankings = compute_elo_rankings(matches_df=df_surface, k_factor=27, surface=surface)

    # Assign the computed rankings back to the original DataFrame.
    df.loc[df['surface_group'] == surface, [feature_p1, feature_p2]] = rankings

    # Calculate and assign the difference between player1 and player2 ELO rankings.
    df.loc[df['surface_group'] == surface, feature_diff] = rankings[:, 0] - rankings[:, 1]

#### 2. ELO - All

In [ ]:
# ELO RANKING
match_elo_rankings, match_momentum = compute_momentum_elo_rankings(matches_df=df.copy(), k_base=70, divisor=200)

# Feature names
type_feature = "elo"
type_feature_sigma = "elo_sigma"

feature_p1 = f"{type_feature}_p1"
feature_p2 = f"{type_feature}_p2"
feature_p1_sigma = f"{type_feature_sigma}_p1"
feature_p2_sigma = f"{type_feature_sigma}_p2"
feature_diff = f"{type_feature}_diff"

# Assign ELO rankings
df[[feature_p1, feature_p2]] = match_elo_rankings
df[feature_diff] = df[feature_p1] - df[feature_p2]

# Assign momentum rankings
df[[feature_p1_sigma, feature_p2_sigma]] = match_momentum

# Print min/max for debugging
print(f"ELO rankings: {match_elo_rankings.min()} - {match_elo_rankings.max()}")
print(f"Momentum: {match_momentum.min()} - {match_momentum.max()}")

#### 3 ELO - Time Decay 

In [ ]:
# col_prefix = "elo"
# match_elo_rankings = compute_time_decay(df_matches=df.copy(), lambda_=0.05, col_prefix=col_prefix)

# type_feature = "elo_time"
# feature1   = f"{type_feature}_p1"
# feature2   = f"{type_feature}_p2"
# minus_name = f"{type_feature}_diff"

# df[[feature1, feature2]] = match_elo_rankings
# df[minus_name] = df[feature1] - df[feature2]

# print(f"{match_elo_rankings.min()} - {match_elo_rankings.max()}")

#### 4 ELO - Movement

In [ ]:
new_features = []

for last_match in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30]:   

    match_movements = compute_rating_movement(matches_df=df.copy(), lookback_matches=last_match, rating_prefix="elo")

    type_feature = f"elo_mov_{last_match}"
    feature_p1   = f"{type_feature}_p1"
    feature_p2   = f"{type_feature}_p2"
    feature_diff = f"{type_feature}_diff"

    # Convert NumPy array to DataFrame with proper column names
    match_df = pd.DataFrame(match_movements, columns=[feature_p1, feature_p2])
    
    # compute difference
    match_df[feature_diff] = match_movements[:, 0] - match_movements[:, 1]

    new_features.append(match_df)

# Concatenate all new columns to the original DataFrame at once
df = pd.concat([df] + new_features, axis=1)


#### 5 ELO - Save df

In [ ]:
# Save
df.to_csv(FILE_TMP)

### C. GLICKO RANKING

In [ ]:
df = pd.read_csv(FILE_TMP)

#### 1. GLICKO - Surface

In [ ]:

type_feature = "glicko_surface"
feature_p1 = f"{type_feature}_p1"
feature_p2 = f"{type_feature}_p2"
feature_diff = f"{type_feature}_diff"

# Initialize new columns in the DataFrame
df[[feature_p1, feature_p2, feature_diff]] = None

for surface in surface_groups:
    # Filter the DataFrame for the current surface group.
    df_surface = df[df['surface_group'] == surface].copy().reset_index(drop=True)
    
    # Compute the GLIKO rankings for the current surface group.
    rankings = compute_glicko_ratings(matches_df=df_surface, surface=surface)

    # Assign the computed rankings back to the original DataFrame.
    df.loc[df['surface_group'] == surface, [feature_p1, feature_p2]] = rankings

    # Calculate and assign the difference between player1 and player2 GLICKO rankings.
    df.loc[df['surface_group'] == surface, feature_diff] = rankings[:, 0] - rankings[:, 1]

#### 2.GLICKO - All

In [ ]:
match_glicko_rankings = compute_glicko_ratings(matches_df=df.copy(), 
                                               initial_rating=1500, 
                                               initial_rating_deviation=400, 
                                               q_factor=0.00068)

type_feature = "glicko"
feature1   = f"{type_feature}_p1"
feature2   = f"{type_feature}_p2"
minus_name = f"{type_feature}_diff"

df[[feature1, feature2]] = match_glicko_rankings
df[minus_name] = df[feature1] - df[feature2]

print(f"{match_glicko_rankings.min()} - {match_glicko_rankings.max()}")

#### 3. GLICKO - Time Decay 

In [ ]:
# col_prefix = "glicko"
# match_elo_rankings = compute_time_decay(df_matches=df.copy(), lambda_=0.05, col_prefix=col_prefix)

# type_feature = "glicko_time"
# feature1   = f"{type_feature}_p1"
# feature2   = f"{type_feature}_p2"
# minus_name = f"{type_feature}_diff"

# df[[feature1, feature2]] = match_elo_rankings
# df[minus_name] = df[feature1] - df[feature2]

# print(f"{match_elo_rankings.min()} - {match_elo_rankings.max()}")

#### 4. GLICKO - Movement

In [ ]:
new_features = []
for last_match in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30]:   

    match_movements = compute_rating_movement(matches_df=df.copy(), lookback_matches=last_match, rating_prefix="glicko")

    type_feature = f"glicko_mov_{last_match}"
    feature_p1   = f"{type_feature}_p1"
    feature_p2   = f"{type_feature}_p2"
    feature_diff = f"{type_feature}_diff"

    # Convert NumPy array to DataFrame with proper column names
    match_df = pd.DataFrame(match_movements, columns=[feature_p1, feature_p2])
    
    # compute difference
    match_df[feature_diff] = match_movements[:, 0] - match_movements[:, 1]

    new_features.append(match_df)

# Concatenate all new columns to the original DataFrame at once
df = pd.concat([df] + new_features, axis=1)

#### 5. GLICKO - Save df

In [ ]:
# Save
df.to_csv(FILE_TMP)

### D. TRUESKILL RANKING

In [ ]:
df = pd.read_csv(FILE_TMP)

#### 1. TRUESKILL - Surface

In [ ]:

type_feature = "trueskill_surface"
feature_p1 = f"{type_feature}_p1"
feature_p2 = f"{type_feature}_p2"
feature_diff = f"{type_feature}_diff"

# Initialize new columns in the DataFrame
df[[feature_p1, feature_p2, feature_diff]] = None

for surface in surface_groups:
    # Filter the DataFrame for the current surface group.
    df_surface = df[df['surface_group'] == surface].copy().reset_index(drop=True)

    # Compute the TRUESKILL rankings for the current surface group.
    rankings = compute_trueskill_ratings(matches_df=df_surface, surface=surface)

    # Assign the computed rankings back to the original DataFrame.
    df.loc[df['surface_group'] == surface, [feature_p1, feature_p2]] = rankings

    # Calculate and assign the difference between player1 and player2 TRUESKILL rankings.
    df.loc[df['surface_group'] == surface, feature_diff] = rankings[:, 0] - rankings[:, 1]

#### 2. TRUESKILL - All

In [ ]:
# TRUESKILL RANKING
match_trueskill_rankings = compute_trueskill_ratings(matches_df=df.copy())

type_feature = "trueskill"
feature1   = f"{type_feature}_p1"
feature2   = f"{type_feature}_p2"
minus_name = f"{type_feature}_diff"

df[[feature1, feature2]] = match_trueskill_rankings
df[minus_name] = df[feature1] - df[feature2]

print(f"{match_trueskill_rankings.min()} - {match_trueskill_rankings.max()}")

#### 3. TRUESKILL - Time Decay 

In [ ]:
# col_prefix = "trueskill"
# match_elo_rankings = compute_time_decay(df_matches=df.copy(), lambda_=0.05, col_prefix=col_prefix)

# type_feature = "trueskill_time"
# feature1   = f"{type_feature}_p1"
# feature2   = f"{type_feature}_p2"
# minus_name = f"{type_feature}_diff"

# df[[feature1, feature2]] = match_elo_rankings
# df[minus_name] = df[feature1] - df[feature2]

# print(f"{match_elo_rankings.min()} - {match_elo_rankings.max()}")

#### 4. TRUESKILL - Movement

In [ ]:
new_features = []
for last_match in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30]:   

    match_movements = compute_rating_movement(matches_df=df.copy(), lookback_matches=last_match, rating_prefix="trueskill")

    type_feature = f"trueskill_mov_{last_match}"
    feature_p1   = f"{type_feature}_p1"
    feature_p2   = f"{type_feature}_p2"
    feature_diff = f"{type_feature}_diff"

    # Convert NumPy array to DataFrame with proper column names
    match_df = pd.DataFrame(match_movements, columns=[feature_p1, feature_p2])
    
    # compute difference
    match_df[feature_diff] = match_movements[:, 0] - match_movements[:, 1]

    new_features.append(match_df)

# Concatenate all new columns to the original DataFrame at once
df = pd.concat([df] + new_features, axis=1)

#### 5.TRUESKILL - Save df

In [ ]:
# Save
df.to_csv(FILE_TMP)

## III. Popularity 

In [ ]:
df = pd.read_csv(FILE_TMP, low_memory=False)

In [ ]:
from prediction_tennis.src.preprocessing.features.popularity_player_features import FactorInclusion, PopularityMethod, PopularityPeriod, compute_player_popularity_before_match

### A. SIMPLE Popularity (TYPE and ROUND)

In [ ]:

df = df.sort_values("timestamp").reset_index(drop=True)

new_features = []

for methods in [PopularityMethod.MULTIPLICATIVE, PopularityMethod.ADDITIVE, PopularityMethod.GEOMETRIC_MEAN, PopularityMethod.HARMONIC_MEAN, PopularityMethod.WEIGHTED_SUM]:
    for factor_inclusion in [FactorInclusion.BOTH, FactorInclusion.TYPE_ONLY, FactorInclusion.ROUND_ONLY]:
        for time_window in [PopularityPeriod.ALL_TIME]: #PopularityPeriod.TWO_YEARS
            match_popularity = compute_player_popularity_before_match(df, methods=methods, factor_inclusion=factor_inclusion, time_window=time_window) 

            # Get method details
            method_name = methods.value
            factor_suffix = factor_inclusion.value
            column_suffix = f"{method_name}_{factor_suffix}"

            type_feature = f"popularity_{column_suffix}"
            feature_p1   = f"{type_feature}_p1"
            feature_p2   = f"{type_feature}_p2"
            feature_diff = f"{type_feature}_diff"

            # Convert NumPy array to DataFrame with proper column names
            match_df = pd.DataFrame(match_popularity, columns=[feature_p1, feature_p2])

            # compute difference
            match_df[feature_diff] = match_movements[:, 0] - match_movements[:, 1]

            new_features.append(match_df)

    print(f"Method: {methods} ✅")

# Concatenate all new columns to the original DataFrame at once
df = pd.concat([df] + new_features, axis=1)


### B Save df

In [ ]:
# Save
df.to_csv(FILE_TMP)

## IV. Match feature

In [ ]:
df = pd.read_csv(FILE_TMP, low_memory=False)

In [ ]:
from prediction_tennis.src.preprocessing.features.match_features import compute_match_features

In [ ]:
df = compute_match_features(combined_flashscore_df=df)

In [ ]:
# Save
df.to_csv(FILE_TMP)

## V. Time feature

In [ ]:
df = pd.read_csv(FILE_TMP, low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])

### A. global value

In [ ]:
from prediction_tennis.src.preprocessing.features.last_match_metrics import compute_match_count_ratios, compute_win_count_ratios

# DAYS
DEFAULT_1_DAYS_IN_SECONDS  : int = 60 * 60 * 24 

# MONTHS
DEFAULT_1_MONTHS_IN_SECONDS: int = DEFAULT_1_DAYS_IN_SECONDS * 30
DEFAULT_2_MONTHS_IN_SECONDS: int = DEFAULT_1_MONTHS_IN_SECONDS * 1
DEFAULT_6_MONTHS_IN_SECONDS: int = DEFAULT_1_MONTHS_IN_SECONDS * 6

# YEARS
DEFAULT_1_YEARS_IN_SECONDS : int = DEFAULT_1_DAYS_IN_SECONDS * 365
DEFAULT_2_YEARS_IN_SECONDS : int = DEFAULT_1_YEARS_IN_SECONDS * 2

# INFINITY
DEFAULT_INFINY_IN_SECONDS : int = DEFAULT_1_YEARS_IN_SECONDS * 10


### B. LAST MATCH played by times (days, month, years, alls)

In [ ]:
# Mapping of time window labels to days
from typing import Dict, List

import numpy as np


TIME_WINDOWS: Dict[str, int] = {
    "1d": 1,
    "2d": 2,
    "3d": 3,
    "4d": 4,
    "5d": 5,
    "6d": 6,
    "7d": 7,
    "10d": 10,
    "15d": 15,
    "20d": 20,
    "25d": 25,
    "1m": 30,
    "2m": 61,
    "4m": 122,
    "6m": 183,
    "1y": 365,
    "all": 365 * 20,
}
new_feature_frames: List[pd.DataFrame] = []

for window_label, period_days in TIME_WINDOWS.items():   
   # Compute ratios for matches and wins
    match_count_ratio = compute_match_count_ratios(matches_df=df.copy(), period_days=period_days, use_ratio=USE_FRACTION)
    win_count_ratio = compute_win_count_ratios(matches_df=df.copy(), period_days=period_days, use_ratio=USE_FRACTION)
    
    # Define feature names
    match_features = {
                     "p1"  : f"{USE_FRACTION_STR}_match_{window_label}_p1",
                     "p2"  : f"{USE_FRACTION_STR}_match_{window_label}_p2",
                     "diff": f"{USE_FRACTION_STR}_match_{window_label}_diff",
                     }

    win_features = {
                   "p1"  : f"{USE_FRACTION_STR}_win_{window_label}_p1",
                   "p2"  : f"{USE_FRACTION_STR}_win_{window_label}_p2",
                   "diff": f"{USE_FRACTION_STR}_win_{window_label}_diff",
                   }

    match_win_features = {
                         "p1"  : f"{USE_FRACTION_STR}_match_win_{window_label}_p1",
                         "p2"  : f"{USE_FRACTION_STR}_match_win_{window_label}_p2",
                         "diff": f"{USE_FRACTION_STR}_match_win_{window_label}_diff",
                         }
    
    # Convert NumPy arrays to DataFrames with descriptive column names
    match_df = pd.DataFrame(match_count_ratio, columns=[match_features["p1"], match_features["p2"]])
    win_df = pd.DataFrame(win_count_ratio, columns=[win_features["p1"], win_features["p2"]])

    # Compute differences between player 1 and player 2
    match_df[match_features["diff"]] =  match_count_ratio[:, 0] - match_count_ratio[:, 1]
    win_df[win_features["diff"]]     = win_count_ratio[:, 0] - win_count_ratio[:, 1]
    
    # Compute match-win features only if use_fraction is False
    if not USE_FRACTION:
        # Avoid division by zero using np.divide with 'where'
        match_win_array = np.divide(win_count_ratio, match_count_ratio, out=np.zeros_like(win_count_ratio, dtype=float), where=match_count_ratio != 0)
        match_win_df = pd.DataFrame(match_win_array,columns=[match_win_features["p1"], match_win_features["p2"]],)

        # Compute difference safely
        diff_col = (np.divide(win_count_ratio[:, 0], match_count_ratio[:, 0], out=np.zeros_like(win_count_ratio[:, 0], dtype=float), where=match_count_ratio[:, 0] != 0) 
                    - 
                    np.divide(win_count_ratio[:, 1], match_count_ratio[:, 1], out=np.zeros_like(win_count_ratio[:, 1], dtype=float), where=match_count_ratio[:, 1] != 0))

        match_win_df[match_win_features["diff"]] = diff_col
        combined_df = pd.concat([match_df, win_df, match_win_df], axis=1)
    else:
        combined_df = pd.concat([match_df, win_df], axis=1)

    new_feature_frames.append(combined_df)

df = pd.concat([df] + new_feature_frames, axis=1)


## VI. TO CSV

In [ ]:
df.to_csv(FILE_FEATURE)